# Generación de Dataset Adversario (Demo de Streaming)

Este notebook toma el dataset original, filtra el inglés, y cruza aleatoriamente los textos de la mitad de los registros.
El objetivo es demostrar visualmente en Streamlit cómo el modelo reacciona ante errores humanos flagrantes (el texto no cuadra con la etiqueta) e interrumpe o redirige la operación.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Carga de datos RAW
file_path = '../../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv'
df = pd.read_csv(file_path)

# 2. Filtrado y aislamiento
df_en = df[df['language'] == 'en'].copy()
df_vital = df_en[['body', 'queue', 'type', 'priority']].copy()

# 3. Renombrado al contrato Pydantic
df_vital = df_vital.rename(columns={
    'body': 'interaction_content',
    'queue': 'Assignment_Group',
    'type': 'Interaction_Type',
    'priority': 'Priority'
})

# Añadimos trazabilidad
df_vital.insert(0, 'ticket_id', [f"TKT-DEMO-{i+1:04d}" for i in range(len(df_vital))])

# Recorte para demo ágil (100 filas)
df_demo = df_vital.head(100).copy()
print(f"Tickets cargados: {len(df_demo)}")

Tickets cargados: 100


In [2]:
# 4. Inyección de Ruido Adversario (Text Swapping)
np.random.seed(42)

# Mutamos la segunda mitad del dataset (50 tickets)
indices_a_mutar = df_demo.index[50:]
textos = df_demo.loc[indices_a_mutar, 'interaction_content'].values
np.random.shuffle(textos)

# Sobrescribimos rompiendo la alineación semántica
df_demo.loc[indices_a_mutar, 'interaction_content'] = textos

print("Cruzamiento de textos completado.")
df_demo.tail(3)

Cruzamiento de textos completado.


C:\Users\andre\AppData\Local\Temp\ipykernel_9144\2419148714.py:7: UserWarning: you are shuffling a 'ArrowStringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(textos)


,ticket_id,interaction_content,Assignment_Group,Interaction_Type,Priority
139,TKT-DEMO-0098,"Dear Customer Support Team,\n\nI am reaching o...",Customer Service,Request,low
140,TKT-DEMO-0099,"Dear Customer Support,\n\nI am submitting a re...",Customer Service,Request,high
141,TKT-DEMO-0100,"Dear Customer Support Team,\n\nI am reaching o...",IT Support,Problem,high


In [3]:
# 5. Volcado al punto de montaje de Streamlit
output_dir = '../../data/resultados/'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'predicciones_holdout_roberta.csv')
df_demo.to_csv(output_path, index=False)

print(f"Archivo listo en: {output_path}")

Archivo listo en: ../../data/resultados/predicciones_holdout_roberta.csv
